In [16]:

import pandas as pd
import numpy as np

# Load your cleaned dataset
data = pd.read_excel("Cleaned_Rheumatic_Lupus_Dataset.xlsx")

# Check class balance
data['Disease'].value_counts()


Disease
Rheumatoid Arthritis            2848
Systemic Lupus Erythematosus    1355
Name: count, dtype: int64

In [17]:
ra = data[data['Disease'] == 'Rheumatoid Arthritis'].copy()
lupus = data[data['Disease'] == 'Lupus'].copy()


In [23]:
def generate_synthetic_data(df, target_size):
    import numpy as np
    import pandas as pd

    num_cols = ['Age', 'ESR', 'CRP', 'RF', 'Anti-CCP', 'C3', 'C4']
    cat_cols = ['Gender', 'HLA-B27', 'ANA', 'Anti-Ro', 'Anti-La', 'Anti-dsDNA', 'Anti-Sm']

    # --- NUMERICAL SYNTHESIS ---
    real_data = df[num_cols].dropna()

    # Replace inf/nan values safely
    mean = real_data.mean().fillna(real_data.median())
    cov = real_data.cov().fillna(0)

    try:
        synthetic_numeric = np.random.multivariate_normal(mean, cov, size=target_size)
    except np.linalg.LinAlgError:
        # Fallback if covariance is singular
        synthetic_numeric = np.zeros((target_size, len(num_cols)))
        for i, col in enumerate(num_cols):
            col_mean = real_data[col].mean()
            col_std = real_data[col].std() if real_data[col].std() > 0 else real_data[col].mean() * 0.1
            synthetic_numeric[:, i] = np.random.normal(col_mean, col_std, target_size)

    synthetic_numeric = pd.DataFrame(synthetic_numeric, columns=num_cols)

    # Ensure realistic biological bounds
    realistic_ranges = {
        "Age": (18, 85),
        "ESR": (0, 120),
        "CRP": (0, 100),
        "RF": (0, 200),
        "Anti-CCP": (0, 200),
        "C3": (50, 180),
        "C4": (10, 80)
    }

    for col, (low, high) in realistic_ranges.items():
        synthetic_numeric[col] = synthetic_numeric[col].clip(lower=low, upper=high)

    # --- CATEGORICAL SYNTHESIS ---
    # --- CATEGORICAL SYNTHESIS ---

    synthetic_categorical = pd.DataFrame()

    for col in cat_cols:
        probs = df[col].value_counts(normalize=True)

    # if the column is completely empty, assume Negative for all
        if probs.empty:
            synthetic_categorical[col] = ["Negative"] * target_size
        else:
            synthetic_categorical[col] = np.random.choice(
            probs.index, size=target_size, p=probs.values
        )


    # Combine all
    synthetic = pd.concat([synthetic_numeric, synthetic_categorical], axis=1)
    synthetic["Disease"] = df["Disease"].iloc[0]

    return synthetic


In [24]:
# ⚙️ Step 3: Generate new rows (~20,000)
target_total = 20000
ra_new = generate_synthetic_data(ra, target_total // 2)
lupus_new = generate_synthetic_data(lupus, target_total // 2)

# Combine old + new data
expanded_data = pd.concat([data, ra_new, lupus_new], axis=0)

# Remove duplicates for safety
expanded_data = expanded_data.drop_duplicates().reset_index(drop=True)

print("✅ Final dataset size:", expanded_data.shape)


C:\Users\navkar enterprises\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\function_base.py:520: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis, **keepdims_kw)
C:\Users\navkar enterprises\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\core\_methods.py:121: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(
C:\Users\navkar enterprises\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\core\frame.py:10869: RuntimeWarning: Degrees of freedom <= 0 for slice
  base_cov = np.cov(mat.T, ddof=ddof)
C:\Users\navkar enterprises\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
C:\Users\navkar enterprises\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


IndexError: single positional indexer is out-of-bounds

In [25]:
import pandas as pd
import numpy as np

# 1️⃣ Load dataset
data = pd.read_excel("Rheumatic and Autoimmune Disease Dataset (1).xlsx")

# 2️⃣ Normalize disease names (handle case, spelling differences)
data['Disease'] = data['Disease'].astype(str).str.lower().str.strip()
data = data[data['Disease'].isin(['rheumatic arthritis', 'rheumatoid arthritis', 'lupus', 'systemic lupus erythematosus'])]

# 3️⃣ Reset index
data.reset_index(drop=True, inplace=True)

# 4️⃣ Fill missing numeric and categorical columns with medically defendable values
for col in data.select_dtypes(include=['number']).columns:
    median_val = data[col].median()
    data[col] = data[col].fillna(median_val)

for col in data.select_dtypes(include=['object']).columns:
    mode_val = data[col].mode()[0] if not data[col].mode().empty else "Unknown"
    data[col] = data[col].fillna(mode_val)

# 5️⃣ Define data generation function
def generate_synthetic_data(df, target_size):
    """Generate realistic, medically consistent synthetic data."""
    from sklearn.covariance import EmpiricalCovariance
    
    numeric_cols = df.select_dtypes(include=['number']).columns
    cat_cols = df.select_dtypes(include=['object']).columns.drop('Disease', errors='ignore')

    # -- Numeric synthesis
    numeric_data = df[numeric_cols]
    mean = numeric_data.mean()
    cov = numeric_data.cov()
    cov = cov.fillna(0)

    # Defensive fix for singular matrices
    try:
        synth_numeric = np.random.multivariate_normal(mean, cov, size=target_size)
    except np.linalg.LinAlgError:
        synth_numeric = np.random.normal(loc=mean, scale=numeric_data.std(), size=(target_size, len(mean)))

    synth_numeric_df = pd.DataFrame(synth_numeric, columns=numeric_cols)

    # -- Categorical synthesis
    synth_cat = pd.DataFrame()
    for col in cat_cols:
        probs = df[col].value_counts(normalize=True)
        if probs.empty:
            synth_cat[col] = ["Negative"] * target_size
        else:
            synth_cat[col] = np.random.choice(probs.index, size=target_size, p=probs.values)

    # -- Add disease label
    disease_name = df["Disease"].iloc[0] if not df.empty else "unknown"
    synth_cat["Disease"] = [disease_name] * target_size

    # Combine numeric + categorical
    synthetic_df = pd.concat([synth_numeric_df, synth_cat], axis=1)
    return synthetic_df

# 6️⃣ Separate RA and Lupus safely
ra = data[data['Disease'].str.contains('rheum', case=False, na=False)]
lupus = data[data['Disease'].str.contains('lupus', case=False, na=False)]

# Defensive fallback if lupus or ra empty
if ra.empty or lupus.empty:
    print("⚠️ Warning: One of the diseases is missing after filtering.")
    print("RA:", len(ra), " | Lupus:", len(lupus))

# 7️⃣ Generate 20k new rows total (10k each)
ra_new = generate_synthetic_data(ra, 10000)
lupus_new = generate_synthetic_data(lupus, 10000)

# 8️⃣ Combine and clean duplicates
expanded_data = pd.concat([data, ra_new, lupus_new], axis=0)
expanded_data.drop_duplicates(inplace=True)
expanded_data.reset_index(drop=True, inplace=True)

print("✅ Final dataset shape:", expanded_data.shape)

# 9️⃣ Save to Excel
expanded_data.to_excel("expanded_medical_dataset.xlsx", index=False)
print("💾 File saved as: expanded_medical_dataset.xlsx")


✅ Final dataset shape: (24203, 15)
💾 File saved as: expanded_medical_dataset.xlsx
